In [126]:
## loading packages
import pandas as pd
import numpy as np
import plotly.graph_objects as go
#import plotly.express as px
import argparse
import sys
import re
import random
import difflib
from collections import defaultdict

import logging

# Configure once (usually at program start)
logging.basicConfig(
    level=logging.INFO,  # minimum level to display
    format="%(asctime)s [%(levelname)s] %(message)s"
)


def is_notebook():
    try:
        from IPython import get_ipython
        shell = get_ipython().__class__.__name__
        if shell == 'ZMQInteractiveShell':
            return True   # Jupyter notebook or qtconsole
        elif shell == 'TerminalInteractiveShell':
            return False  # Terminal running IPython
        else:
            return False  # Other type (?)
    except Exception:
        return False      # Probably standard Python interpreter

In [125]:
## argparse definition
if not is_notebook():
    script_name = sys.argv[0]
else:
    script_name = 'Script'

parser = argparse.ArgumentParser(description=f"{script_name} parameters")

## eigenvectors
parser.add_argument('--eigenvec', type=str, default=None, required=False, help='Eigenvec file path (plink)')
parser.add_argument('--eigenvecID', type=str, default=None, help='Eigenvec ID column (default first column)')
parser.add_argument('--prefix', type=str, default='C', help='Prefix for the dimension in the eigenvec file')

## eigenvalues
parser.add_argument('--eigenval', type=str, default=None, required=False, help='Eigenval file path (plink)')
#parser.add_argument('--reduction', type=str, default='MDS', help='Reduction method')
parser.add_argument('--nb_eigenvalues', type=int, default=10, help='Number of eigenvalues to show (0 for all)')

## stats
parser.add_argument('--imiss', type=str, default=None, help='Imiss file path (plink)')
parser.add_argument('--lmiss', type=str, default=None, help='Lmiss file path (plink)')
parser.add_argument('--frq', type=str, default=None, help='Frq file path (plink)')

## annotation
parser.add_argument('--annotation', type=str, default=None, help='Annotation file path')
parser.add_argument('--annotationID', type=str, default='Genetic ID', help='Annotation ID column (default first column)')
parser.add_argument('--longitude', type=str, default=None, help='Longitude column name')
parser.add_argument('--latitude', type=str, default=None, help='Latitude column name')
parser.add_argument('--time', type=str, default=None, help='Time column name')
#parser.add_argument('--group', type=str, default=None, help='Grouping column name')

## text handling
parser.add_argument('--ignore_case', action='store_true', default=None, help='Ignore case differences in annotation')
parser.add_argument('--no_ignore_case', dest='ignore_case', default=None, action='store_true', help='Do not ignore case differences in annotation')
parser.set_defaults(ignore_case=True)

parser.add_argument('--ignore_space', action='store_true', default=None, help='Ignore space differences in annotation')
parser.add_argument('--no_ignore_space', dest='ignore_space', default=None, action='store_true', help='Do not ignore space differences in annotation')
parser.set_defaults(ignore_space=True)

parser.add_argument('--col_abbrev', type=int, default=15, help='Abbreviate column names to this length (0 for no abbreviation)')
parser.add_argument('--legend_abbrev', type=int, default=0, help='Abbreviate legend text to this length (0 for no abbreviation)')

parser.add_argument('--max_factors', type=int, default=400, help='Maximum number of different elements in factorial columns to be included')

## time figure
parser.add_argument('--time_hist', action='store_true', default=None, help='Show points in time as histogram')
parser.add_argument('--time_scatter', dest='time_hist', action='store_true', default=None, help='Show points in time as scatter plot')
parser.set_defaults(time_hist=True)
parser.add_argument('--time_hist_nbins', type=int, default=100, help='Number of bins for the tme histogram (100)')

## server settings
parser.add_argument('--use_server', action='store_true', default=None, help='Use a dash server for interactive plots')
parser.add_argument('--no_server', dest='use_server', default=None, action='store_false', help='Do not use a dash server: less interactive but static HTML output')
parser.set_defaults(use_server=True)

parser.add_argument('--open_browser', action='store_true', default=None, help='Open directly the dash server in a web browser')
parser.add_argument('--no_open_browser', dest='open_browser', default=None, action='store_false', help='Do not open the dash server in a web browser: you will have to open it manually')
parser.set_defaults(open_browser=False)

parser.add_argument('--server_port', type=int, default=8050, help='Port for the dash server (default 8050)')

## self-contained HTML
parser.add_argument('--html', action='store_true', default=None, help='Generate self-contained HTML')
parser.add_argument('--no_html', dest='html', action='store_true', default=None, help='Do not generate self-contained HTML')
parser.set_defaults(html=False)
parser.add_argument('--html_file', type=str, default=None, help='File name for self-contained HTML (default: eigenvec file name + .html)')


## development
parser.add_argument('--dev', action='store_true', default=None, help='Use development parameters')
parser.add_argument('--no_dev', dest='dev', action='store_false', default=None, help='Do not use development parameters')
parser.set_defaults(dev=False)

#parser.print_help()

In [127]:

## development arguments
# Parse NOTHING when running in a notebook
if is_notebook():
    args = parser.parse_args([])      # <— key line ([])
else:
    args = parser.parse_args()

if is_notebook() or args.dev:
    args.eigenvec='aadr.eigenvec'
    args.eigenval='aadr.eigenval'

    args.imiss='aadr.imiss'
    args.lmiss='aadr.lmiss'
    args.frq='1000gp.frq'

    args.annotation='aadr.anno'
    
    args.reduction='MDS'
    args.eigenvecID='sample'
    args.annotationID='Genetic ID'
    args.nb_eigenvalues=10
    args.prefix='C'
    args.longitude='Long.'
    args.latitude='Lat.'
    args.time='Date mean in BP in years before 1950 CE [OxCal mu for a direct radiocarbon date, and average of range for a contextual date]'
    args.group='Political_Entit'

    args.ignore_case=True
    args.ignore_space=True

    args.col_abbrev=15
    args.legend_abbrev=20

    args.max_factors=400

    args.use_server=False
    args.open_browser=False
    args.server_port=8050
    args.time_hist_nbins=500



if not args.dev and not args.eigenvec:
    logging.error('Missing argument: --eigenvec  is required.')

In [128]:
## functions

## abbreviate a list of strings to a maximum length, preserving uniqueness
def make_unique_abbr(cur_list, max_length=3):
    # Compile regex patterns once
    clean_re = re.compile(r'[^a-zA-Z0-9 ]')
    space_re = re.compile(r' ')

    # Clean and abbreviate
    cleaned = [clean_re.sub('', elem) for elem in cur_list]
    abbreviated = [space_re.sub('_', elem[:max_length]).rstrip('_') for elem in cleaned]

    # Ensure uniqueness using a counter
    counter = defaultdict(int)
    unique_abbr = []

    for abbr in abbreviated:
        new_abbr = abbr
        while new_abbr in counter:
            counter[abbr] += 1
            new_abbr = f"{abbr}{counter[abbr]}"
        counter[new_abbr] = 0
        unique_abbr.append(new_abbr)

    return unique_abbr


## abbreviate columns of a pandas DataFrame
def make_unique_abbr_of_df(df, cols, max_length=3):
    """
    Abbreviate the values in the specified columns of a DataFrame.
    Each unique value in the column is replaced by a unique abbreviation.
    """
    for col in cols:
        unique_vals = df[col].astype(str).unique()
        abbrs = make_unique_abbr(unique_vals, max_length)
        abbr_map = dict(zip(unique_vals, abbrs))
        df[col] = df[col].astype(str).map(abbr_map)
    return df


## search the the best text match within a list of words
def get_abbr_of(target, lookup, returns=None):
    closest = difflib.get_close_matches(target, lookup, n=1)
    if closest is None or len(closest) == 0:
        print(f"Warning: No match found for {target} in {lookup}")
        return None
    
    if returns is  None:
        match = closest[0]
    else:
        match = returns[lookup.index(closest[0])]

    #print(f"Look up for {target} => {match}")
    return match



## de-duplex columns ignoring capitalization and space differences (keep first version of each)
def deduplicate_columns(df, columns, ignore_case=True, ignore_space=True):
    for col in columns:
        norm_col = f'_norm_{col}'
        series = df[col].astype(str)
        if ignore_case:
            series = series.str.lower()
        if ignore_space:
            series = series.str.replace(r'\s+', '', regex=True)
        df[norm_col] = series
        first_map = df.drop_duplicates(norm_col, keep='first').set_index(norm_col)[col]
        df[col] = df[norm_col].map(first_map)
        df.drop(columns=[norm_col], inplace=True)
    return df

## find automatically the pca dimensions prefix
def find_incrementing_prefix_series(columns):
    # Match prefix + number, e.g., PC1, PC2, Dim3
    pattern = re.compile(r'^([A-Za-z_]+)(\d+)$')
    prefix_groups = defaultdict(list)

    # Group columns by prefix
    for col in columns:
        match = pattern.match(col)
        if match:
            prefix, num = match.groups()
            prefix_groups[prefix].append(int(num))

    # Find prefixes with longest incrementing series
    longest_series = []
    for prefix, nums in prefix_groups.items():
        nums_sorted = sorted(nums)
        # Check if numbers form a consecutive sequence
        if nums_sorted == list(range(nums_sorted[0], nums_sorted[-1] + 1)):
            series = [f"{prefix}{n}" for n in nums_sorted]
            if len(series) > len(longest_series):
                longest_series = series

    return longest_series

In [129]:
## reading imiss
if args.imiss is not None:
    logging.info(f"Reading imiss file '{args.imiss}' ...")
    df_imiss = pd.read_csv(args.imiss, sep=r"\s+", skiprows=[1])
    logging.info(f"Reading imiss file '{args.imiss}' ... done.")
else:
    df_imiss = None

2025-09-09 07:47:00,829 [INFO] Reading imiss file 'aadr.imiss' ...
2025-09-09 07:47:00,854 [INFO] Reading imiss file 'aadr.imiss' ... done.


In [130]:
## reading lmiss
if args.lmiss is not None:
    logging.info(f"Reading lmiss file '{args.lmiss}' ...")
    df_lmiss = pd.read_csv(args.lmiss, sep=r"\s+", skiprows=[1], low_memory=False)
    logging.info(f"Reading lmiss file '{args.lmiss}' ... done.")
else:
    df_lmiss = None

2025-09-09 07:47:03,858 [INFO] Reading lmiss file 'aadr.lmiss' ...
2025-09-09 07:47:05,060 [INFO] Reading lmiss file 'aadr.lmiss' ... done.


In [131]:
## reading frq
if args.frq is not None:
    logging.info(f"Reading frq file '{args.frq}' ...")
    df_frq = pd.read_csv(args.frq, sep=r"\s+")
    logging.info(f"Reading frq file '{args.frq}' ... done.")
else:
    df_frq = None


2025-09-09 07:47:25,541 [INFO] Reading frq file '1000gp.frq' ...
2025-09-09 07:47:25,554 [INFO] Reading frq file '1000gp.frq' ... done.


In [141]:
## reading eigenval

if args.eigenval is not None:
    logging.info(f"Reading eigenval file '{args.eigenval}' ...")
    eigenval = pd.read_csv(args.eigenval, sep="\t", header=None, names=["eigenvalue"])

    logging.info(f"   Found {len(eigenval)} eigenvalues.")

    ## compute variance explained
    eigenval["eigenvalue"] = eigenval["eigenvalue"] / eigenval["eigenvalue"].sum()

    ## add cumulative eigenvalues
    eigenval["cumulative"] = eigenval["eigenvalue"].cumsum()

    ## add index starting from 1 in the first column
    eigenval["dimension"] = eigenval.index + 1

    ## order columns
    eigenval = eigenval[["dimension"] + [col for col in eigenval.columns if col != "dimension"]]
else:
    eigenval = None

logging.info(f"Reading eigenval file '{args.eigenval}' ... done.")

2025-09-09 07:54:15,785 [INFO] Reading eigenval file 'aadr.eigenval' ...
2025-09-09 07:54:15,790 [INFO]    Found 10442 eigenvalues.
2025-09-09 07:54:15,793 [INFO] Reading eigenval file 'aadr.eigenval' ... done.


In [140]:
## reading eigenvec (always present)

# Read eigenvectors
logging.info(f"Reading eigenvec file '{args.eigenvec}' ...")
eigenvec = pd.read_csv(args.eigenvec, sep=r"\s+", header=0)

PCS = find_incrementing_prefix_series(eigenvec.columns)

if len(PCS) < 2:
    logging.error('Not enough dimension columns found in the eigenvec file ({len(PCS)} found)).')

logging.info(f"   Found {len(PCS)} principal components ({", ".join(PCS[:2])}, ...).")

# Scale PC columns: to be removed
nb_snp = 194926 # Number of SNPs
eigenvec[PCS] = eigenvec[PCS].div(nb_snp)

logging.info(f"Reading eigenvec file '{args.eigenvec}' ... done.")


2025-09-09 07:54:00,376 [INFO] Reading eigenvec file 'aadr.eigenvec' ...
2025-09-09 07:54:00,418 [INFO]    Found 20 principal components (C1, C2, ...).
2025-09-09 07:54:00,421 [INFO] Reading eigenvec file 'aadr.eigenvec' ... done.


In [142]:
## reading annotation
annotation = None
ANNOTATION_TIME = None
ANNOTATION_LAT = None
ANNOTATION_LONG = None

if args.annotation is not None:
    logging.info(f"Reading annotation file '{args.annotation}' ...")
    annotation = pd.read_csv(args.annotation, sep='\t', na_values='..', low_memory=False)
    
    # create abbreviation_desc data.frame
    annotation_desc = pd.DataFrame({
        'Abbreviation': make_unique_abbr(annotation.columns, max_length=args.col_abbrev),
        'Description': annotation.columns,
        'Type': [annotation[col].dtype.name for col in annotation.columns]
    })

    # Add a column for number of unique elements if the column is categorical
    n_unique = []
    for col in annotation.columns:
        dtype = annotation[col].dtype.name
        if dtype in ['object', 'category']:
            n_unique.append(annotation[col].nunique(dropna=False))
        else:
            n_unique.append(None)
    annotation_desc['N_levels'] = n_unique
    
    # Rename columns of annotation data.frame with the abbreviations
    annotation.columns = annotation_desc['Abbreviation']

    # Get abbreviations for given IDs
    if args.annotationID is not None:
        ANNOTATION_ID = get_abbr_of(args.annotationID, annotation_desc['Description'].to_list(), annotation_desc['Abbreviation'].to_list())
        if ANNOTATION_ID not in annotation.columns:
            logging.info(f"Warning: Specified annotationID '{--annotationID}' not found in annotation columns. Using first column instead.")
    else:
        ANNOTATION_ID = annotation.columns[0] ## default is the first column

    if args.time is not None:
        ANNOTATION_TIME = get_abbr_of(args.time, annotation_desc['Description'].to_list(), annotation_desc['Abbreviation'].to_list())
        if ANNOTATION_TIME not in annotation.columns:
            ANNOTATION_TIME = None
            logging.warning(f"Warning: Specified time '{--time}' not found in annotation columns. Time graph disabled.")

    if args.longitude is not None:
        ANNOTATION_LONG = get_abbr_of(args.longitude, annotation_desc['Description'].to_list(), annotation_desc['Abbreviation'].to_list())
        if ANNOTATION_LONG not in annotation.columns:
            ANNOTATION_LONG = None
            logging.warning(f"Warning: Specified longitude '{--longitude}' not found in annotation columns. Geographical map disabled.")

    if args.latitude is not None:
        ANNOTATION_LAT = get_abbr_of(args.latitude, annotation_desc['Description'].to_list(), annotation_desc['Abbreviation'].to_list())
        if ANNOTATION_LAT not in annotation.columns:
            ANNOTATION_LAT = None
            logging.warning(f"Warning: Specified latitude '{--latitude}' not found in annotation columns. Geographical map disabled.")

    ## clean factorial elements if needed
    exclude_abbr = [elem for elem in [ANNOTATION_ID, ANNOTATION_TIME, ANNOTATION_LONG, ANNOTATION_LAT] if elem != None]

    FACTORIAL_COLUMNS = annotation_desc.loc[
        (annotation_desc['N_levels'].notnull()) &
        (annotation_desc['N_levels'] <= args.max_factors) &
        (~annotation_desc['Abbreviation'].isin(exclude_abbr)),
        'Abbreviation'
    ].tolist()
    
    CONTINUOUS_COLUMNS = [col for col in annotation.columns if col not in FACTORIAL_COLUMNS]
    
    if args.ignore_case or args.ignore_space:
        if args.ignore_case:
            logging.info(f"   Ignoring case differences.")
        if args.ignore_space:
            logging.info(f"   Ignoring space differences.")
        annotation = deduplicate_columns(annotation, FACTORIAL_COLUMNS, args.ignore_case, args.ignore_space)

    if args.legend_abbrev > 0:
        logging.info(f"   Shortening legend text to max {args.legend_abbrev} characters.")
        annotation = make_unique_abbr_of_df(annotation, FACTORIAL_COLUMNS, args.legend_abbrev)

    ## logging
    logging.info(f"   Found {len(FACTORIAL_COLUMNS)} factorial columns.")
    logging.info(f"   Found {len(CONTINUOUS_COLUMNS)} continuous columns.")
    if ANNOTATION_LAT and ANNOTATION_LONG:
        logging.info(f"   Found geographic coordinates columns (latitude/longitude): '{ANNOTATION_LAT}', '{ANNOTATION_LONG}'.")
    if ANNOTATION_TIME:
        logging.info(f"   Found time column: '{ANNOTATION_TIME}'.")
    
    logging.info(f"Reading annotation file '{args.annotation}' ... done.")

2025-09-09 07:55:08,314 [INFO] Reading annotation file 'aadr.anno' ...
2025-09-09 07:55:08,450 [INFO]    Ignoring case differences.
2025-09-09 07:55:08,450 [INFO]    Ignoring space differences.
2025-09-09 07:55:08,603 [INFO]    Shortening legend text to max 20 characters.
2025-09-09 07:55:08,630 [INFO]    Found 10 factorial columns.
2025-09-09 07:55:08,631 [INFO]    Found 32 continuous columns.
2025-09-09 07:55:08,631 [INFO]    Found geographic coordinates columns (latitude/longitude): 'Lat', 'Long'.
2025-09-09 07:55:08,632 [INFO]    Found time column: 'Date_mean_in_BP'.
2025-09-09 07:55:08,632 [INFO] Reading annotation file 'aadr.anno' ... done.


In [143]:
## merging coord and annotation

# Check if annotation exists (not None and not empty)
if annotation is not None and not annotation.empty:
    EIGENVEC_ID = get_abbr_of(args.eigenvecID, eigenvec.columns.to_list()) if args.eigenvecID is not None else eigenvec.columns[0] ## default is the first column

    # Rename EIGENVEC_ID column in annotation to match EIGENVEC_ID in eigenvec
    # so we can join by that column name
    annotation_renamed = annotation.rename(columns={ANNOTATION_ID: EIGENVEC_ID})
    
    # Perform left join
    coord = eigenvec.merge(annotation_renamed, on=EIGENVEC_ID, how="left")
    
    # Negate the values in the ANNOTATION_TIME column
    coord[ANNOTATION_TIME] = -coord[ANNOTATION_TIME]
else:
    coord = eigenvec

# Rename the EIGENVEC_ID column to id
coord = coord.rename(columns={EIGENVEC_ID: "id"})

# Move 'id' column to the front
cols = ['id'] + [c for c in coord.columns if c != 'id']
coord = coord[cols]


In [144]:
## init variables

init_x = 'PC1'
init_y = 'PC2'

sizes = [4, 8, 12, 16]
init_size = 8

In [ ]:
import dash
from dash import dcc, html, Input, Output, State, ctx
from dash_ag_grid import AgGrid
import plotly.express as px
import pandas as pd
from functools import lru_cache
from dash import callback_context


# Replace these with your actual variables
df = coord  # Assumed to be preloaded

def get_annotation_table():
    # Get the column names of coord (PCS at the end)
    all_columns = [col for col in coord if col not in PCS] + PCS

    # Use a dict for fast lookup instead of merge
    abbr_map = {abbr: desc for abbr, desc in zip(annotation_desc['Abbreviation'], annotation_desc['Description'])}
    type_map = {abbr: dtype for abbr, dtype in zip(annotation_desc['Abbreviation'], annotation_desc['Type'])}
    nlevels_map = {abbr: nlev for abbr, nlev in zip(annotation_desc['Abbreviation'], annotation_desc['N_levels'])}

    # Build the table rows directly
    rows = []
    for abbr in all_columns:
        rows.append({
            'Abbreviation': abbr,
            'Description': abbr_map.get(abbr, ''),
            'Type': type_map.get(abbr, ''),
            'N_levels': nlevels_map.get(abbr, '')
        })
    return pd.DataFrame(rows)

extended_annotation_table = get_annotation_table()

##-----------------------------------------------------------------------------
## Annotation tab
def get_annotation_tab():
    if annotation is None:
        return None

    return html.Div(
        id='annotation_tab_content',
        style={'height': '100%', 'width': '100%', 'display': 'flex', 'flexDirection': 'column'},
        children=[
            html.Div([
                AgGrid(
                    id='annotation-table',
                    rowData=extended_annotation_table.to_dict('records'),
                    columnDefs=[
                        {'headerName': '', 'checkboxSelection': True, 'headerCheckboxSelection': True, 'width': 40},
                        *[
                            {
                                'headerName': col,
                                'field': col,
                                'sortable': True,
                            }
                            for col in extended_annotation_table.columns
                        ]
                    ],
                    dashGridOptions={
                        "rowSelection": "multiple",
                        "rowMultiSelectWithClick": True,
                        "pagination": True,
                        "paginationAutoPageSize": True,
                        "defaultColDef": {
                            "resizable": True,
                            "wrapText": True,
                            "autoHeight": True,
                        },
                    },
                    className="ag-theme-alpine",
                    selectedRows=extended_annotation_table.to_dict('records')[:10],
                    style={'height': '100%', 'width': '100%'}
                )
            ], style={'flex': 1, 'height': '100%', 'width': '100%'})
        ]
    )

##-----------------------------------------------------------------------------
## Eigenvalues tab
def get_eigenvalues_tab():
    if args.eigenval is None:
        return None

    return html.Div([
        html.Div([
            dcc.Graph(
                id='eigenvals',
                figure={
                    'data': [{
                        'x': eigenval["dimension"][:args.nb_eigenvalues],
                        'y': 100 * eigenval['eigenvalue'][:args.nb_eigenvalues],
                        'type': 'bar',
                        'name': 'Eigenvalues'
                    }],
                    'layout': {
                        'title': {'text': 'Eigenvalues'},
                        'xaxis': {'title': {'text': 'Dimension'}},
                        'yaxis': {'title': {'text': '% explained variance'}},
                        'autosize': True,
                        'height': None,
                    }
                },
                style={'width': '50%', 'height': 'calc(100vh - 50px - 50px)', 'display': 'inline-block', 'verticalAlign': 'top'}
            ),
            dcc.Graph(
                id='eigenvals_cumulative',
                figure={
                    'data': [{
                        'x': eigenval["dimension"][:args.nb_eigenvalues],
                        'y': 100 * eigenval['cumulative'][:args.nb_eigenvalues],
                        'type': 'bar',
                        'name': 'Cumulative'
                    }],
                    'layout': {
                        'title': {'text': 'Cumulative Eigenvalues'},
                        'xaxis': {'title': {'text': 'Dimension'}},
                        'yaxis': {'title': {'text': '% cumulative explained variance'}},
                        'autosize': True,
                        'height': None,
                    }
                },
                style={'width': '50%', 'height': 'calc(100vh - 50px - 50px)', 'display': 'inline-block', 'verticalAlign': 'top'}
            ),
        ], style={
            'width': '100%',
            'height': '100%',
            'display': 'flex',
            'flexDirection': 'row',
            'justifyContent': 'space-between',
            'alignItems': 'stretch'
        }),
    ], id='eigenvalues_tab_content', style={'height': '100vh', 'width': '100%'})


##-----------------------------------------------------------------------------
## Statistics tab
def get_statistics_tab():
    if df_imiss is None and df_lmiss is None and df_frq is None:
        return None  # Don't render the tab at all

    graphs = []

    ## imiss
    if df_imiss is not None:
        fig_imiss = px.histogram(
            df_imiss, x='F_MISS', nbins=50, title='Sample missing rate',
            labels={'F_MISS': 'Missing rate', 'count': 'Count'},
            color_discrete_sequence=['#1F77B4']
        )
        fig_imiss.update_layout(
            template='plotly_white',
            xaxis=dict(range=[0, 1]), 
            autosize=True
        )
        graphs.append(
            dcc.Graph(id='imiss', figure=fig_imiss, style={
                'flex': 1, 'height': 'calc(100vh - 50px - 50px)', 'width': '100%',
                'display': 'inline-block', 'verticalAlign': 'top'
            })
        )

    ## lmiss
    if df_lmiss is not None:
        fig_lmiss = px.histogram(
            df_lmiss, x='F_MISS', nbins=50, title='SNP missing rate',
            labels={'F_MISS': 'Missing rate', 'count': 'Count'},
            color_discrete_sequence=['#1F77B4']
        )
        fig_lmiss.update_layout(
            template='plotly_white',
            xaxis=dict(range=[0, 1]), 
            autosize=True
        )
        graphs.append(
            dcc.Graph(id='lmiss', figure=fig_lmiss, style={
                'flex': 1, 'height': 'calc(100vh - 50px - 50px)', 'width': '100%',
                'display': 'inline-block', 'verticalAlign': 'top'
            })
        )

    ## freq
    if df_frq is not None:
        fig_frq = px.histogram(
            df_frq, x='MAF', nbins=500, title='Minor Allele Frequency',
            labels={'MAF': 'Allele frequency', 'count': 'Count'},
            color_discrete_sequence=['#1F77B4']
        )
        fig_frq.update_layout(
            template='plotly_white',
            xaxis=dict(range=[0, 0.5]), 
            autosize=True
        )
        graphs.append(
            dcc.Graph(id='freq', figure=fig_frq, style={
                'flex': 1, 'height': 'calc(100vh - 50px - 50px)', 'width': '100%',
                'display': 'inline-block', 'verticalAlign': 'top'
            })
        )

    return html.Div([
        html.Div(graphs, style={
            'width': '100%',
            'height': '100%',
            'display': 'flex',
            'flexDirection': 'row',
            'justifyContent': 'space-between',
            'alignItems': 'stretch'
        }),
    ], id='statistics_tab_content', style={'height': '100vh', 'width': '100%'})


##-----------------------------------------------------------------------------
## Help tab
def get_help_tab():
    return html.Div([
        html.Pre(parser.format_help())
    ], id = 'help_tab_content')


##-----------------------------------------------------------------------------
## PCA tab

def get_selectors():
    dropdowns = []

    # X-axis
    dropdowns.append(
        html.Div([
            html.Div("X-axis:", style={"marginRight": "5px"}),
            dcc.Dropdown(id='x-axis-selector', 
                         options=PCS, 
                         value=init_x,
                         clearable=False, 
                         style={'width': '120px'}),
        ], style={'display': 'flex', 'alignItems': 'center', 'marginRight': '20px'})
    )

    # Y-axis
    dropdowns.append(
        html.Div([
            html.Div("Y-axis:", style={"marginRight": "5px"}),
            dcc.Dropdown(id='y-axis-selector', 
                         options=PCS, 
                         value=init_y,
                         clearable=False, 
                         style={'width': '120px'}),
        ], style={'display': 'flex', 'alignItems': 'center', 'marginRight': '20px'})
    )

    # Grouping (optional)
    if FACTORIAL_COLUMNS is not None:
        dropdowns.append(
            html.Div([
                html.Div("Grouping:", style={"marginRight": "5px"}),
                dcc.Dropdown(id='group-selector', 
                             options=FACTORIAL_COLUMNS,
                             value=FACTORIAL_COLUMNS[0],
                             clearable=False, 
                             style={'width': '200px'}),
            ], style={'display': 'flex', 'alignItems': 'center', 'marginRight': '20px'})
        )

    # Point size
    dropdowns.append(
        html.Div([
            html.Div("Point size:", style={"marginRight": "5px"}),
            dcc.Dropdown(id='point-size-selector', 
                         options=sizes, 
                         value=8,
                         clearable=False, 
                         style={'width': '120px'}),
        ], style={'display': 'flex', 'alignItems': 'center'})
    )

    return html.Div(dropdowns, style={'display': 'flex', 'flexWrap': 'wrap', 'marginBottom': '15px'})



##-----------------------------------------------------------------------------
## PCA tab

def get_data_table():
    selected_columns = extended_annotation_table['Abbreviation'].head(10).tolist()
    defs = [{'headerName': col, 'field': col, 'sortable': True, 'resizable': True, 
             **({'checkboxSelection': True, 'headerCheckboxSelection': True} if col == 'id' else {})} 
             for col in selected_columns]

    return html.Div(
        AgGrid(
            id='data-table',
            rowData=df.to_dict('records'),
            columnDefs=defs,
            selectedRows=df.to_dict('records'),
            dashGridOptions={
                "rowSelection": "multiple",
                "pagination": True,
                "paginationAutoPageSize": True
            },
            className="ag-theme-alpine",
            style={'height': '100%', 'width': '100%', 'overflowX': 'auto'}
        ),
        style={'height': '50%', 'width': '100%'}
    )

def get_pca_tab():
    # LEFT column children (always present)
    left_children = [
        get_selectors(),
        dcc.Graph(id='pca-plot', style={'height': 'calc(100% - 10cm)', 'width': '100%'})
    ]
    if ANNOTATION_TIME is not None:
        hist_fig = px.histogram(
            df, x=ANNOTATION_TIME, nbins=100, title="", color_discrete_sequence=['#1F77B4']
        )
        hist_fig.update_layout(
            template='plotly_white',
            showlegend=False,
            xaxis=dict(
                rangeslider=dict(visible=True),
                type="linear" 
            )
        )
    
        left_children.append(
            dcc.Graph(id='time-histogram', figure=hist_fig, style={'height': '10cm', 'width': '100%'})
        )

    # RIGHT column children (optional)
    right_children = []
    if ANNOTATION_LAT is not None and ANNOTATION_LONG is not None:
        ## map plot
        right_children.append(
            html.Div(
                dcc.Graph(id='map-plot', style={'height': '100%', 'width': '100%'}),
                style={'height': '50%', 'width': '100%'}
            )
        )

    if annotation is not None:
        ## data table
        right_children.append(get_data_table())

        ## filter
        right_children.append(
            html.Div([
                dcc.Textarea(id='texarea-expression', style={'width': '100%', 'height': '80px'}),
            ])
        )

    # Columns: LEFT always present, RIGHT optional
    columns_children = [
        html.Div(left_children,
                 style={'flex': 1, 'padding': '10px', 'minWidth': 0,
                        'display': 'flex', 'flexDirection': 'column'})
    ]
    if right_children:
        columns_children.append(
            html.Div(right_children,
                     style={'flex': 1, 'padding': '10px', 'minWidth': 0,
                            'display': 'flex', 'flexDirection': 'column'})
        )

    # Main tab layout
    return html.Div(
        id='pca_tab_content',
        children=[
            html.Div(
                id='selection-count',
                style={'padding': '10px', 'fontWeight': 'bold'},
                children=f"Showing all {len(df)} points"
            ),
            html.Div(columns_children,
                     style={'display': 'flex', 'width': '100%', 'height': 'calc(100vh - 50px - 100px)'}
            )
        ]
    )







def labeled_tabs():
    return html.Div([
        html.Div("interactivePCA", style={'marginRight': '40px', 'whiteSpace': 'nowrap', 'fontWeight': 'bold', 'fontSize': 50}),
        dcc.Tabs(
            id="tabs",
            value='pca_tab',
            children=[
                dcc.Tab(label='PCA', value='pca_tab'),
                dcc.Tab(label='Annotation', value='annotation_tab'),
                dcc.Tab(label='Eigenvalues', value='eigenvalues_tab'),
                dcc.Tab(label='Statistics', value='statistics_tab'),
                dcc.Tab(label='Help', value='help_tab')
            ],
            style={'flexGrow': 1, 'height': '50px', 'fontWeight': 'bold', 'justifyContent': 'center', 'alignItems': 'center'}
        )
    ], style={
        'display': 'flex',
        'alignItems': 'center',
        'marginBottom': '20px',
        'height': '50px',
    })



###---------------------------------------------------
## main layout
app = dash.Dash(__name__)



def labeled_tabs(visible_tab_ids):
    tab_labels = {
        'pca_tab': 'PCA',
        'annotation_tab': 'Annotation',
        'eigenvalues_tab': 'Eigenvalues',
        'statistics_tab': 'Statistics',
        'help_tab': 'Help'
    }

    return html.Div(
        style={'display': 'flex', 'alignItems': 'center', 'width': '100%'},
        children=[
            # Left title
            html.Div(
                "interactivePCA",
                style={
                    'fontWeight': 'bold',
                    'marginRight': '20px',
                    'fontSize': '20px',
                    'whiteSpace': 'nowrap'
                }
            ),

            # Tabs
            html.Div(
                style={'flex': 1},  # make tabs container take remaining space
                children=dcc.Tabs(
                    id='tabs',
                    value=visible_tab_ids[0] if visible_tab_ids else None,
                    style={'width': '100%'},  # stretch container
                    children=[
                        dcc.Tab(
                            label=tab_labels[tab_id],
                            value=tab_id,
                            style={
                                'flex': 1,              # each tab stretches equally
                                'textAlign': 'center'
                            },
                            selected_style={
                                'flex': 1,
                                'textAlign': 'center',
                                'fontWeight': 'bold'
                            }
                        )
                        for tab_id in visible_tab_ids
                    ]
                )
            )
        ]
    )


# Build tabs dynamically
tab_contents = [
    ('pca_tab', get_pca_tab()),
    ('annotation_tab', get_annotation_tab()),
    ('eigenvalues_tab', get_eigenvalues_tab()),
    ('statistics_tab', get_statistics_tab()),
    ('help_tab', get_help_tab())
]

# Filter out tabs where the *content* is None
tab_contents = [(tab_id, content) for tab_id, content in tab_contents if content is not None]

# Extract IDs and contents
tab_ids = [tab_id for tab_id, _ in tab_contents]
tab_components = [content for _, content in tab_contents]

app.layout = html.Div([
    dcc.Store(id='selected-indices', data=df['id'].tolist()),
    dcc.Store(id='visible-columns', data=extended_annotation_table['Abbreviation'].head(10).tolist()),
    labeled_tabs(tab_ids),
    html.Div(id='tabs-content', children=tab_components)
])




###---------------------------------------------------
## Callback

# Callback to switch tab content
@app.callback(
    [Output(f"{tab_id}_content", "style") for tab_id in tab_ids],
    Input("tabs", "value")
)
def display_tab_content(active_tab):
    return [
        {'display': 'block'} if tab_id == active_tab else {'display': 'none'}
        for tab_id in tab_ids
    ]


## -------------------------------------------------------------------
###  shared figure helpers
def split_selected_unselected(df, selected_ids):
    selected = df[df['id'].isin(selected_ids)]
    unselected = df[~df['id'].isin(selected_ids)]
    return selected, unselected

## hover text
def get_hover_text(columns):

    lines = [f"<b>{col}</b>: %{{customdata[{i}]}}" for i, col in enumerate(columns)]

    # insert an extra empty line after the first element
    if len(lines) >= 1:
        lines.insert(1, "")

    return "<br>".join(lines) + "<extra></extra>"

## hover text
def get_hover_text_minimal():
    return f"{id}: %{{id}}<br><extra></extra>"

## get a unique colour pallet
def get_color_map(df, color_by):
    if color_by not in df.columns:
        raise ValueError(f"Column '{color_by}' not found in DataFrame.")

    # Drop NaNs and get sorted unique values
    unique_values = sorted(df[color_by].dropna().unique())

    if not unique_values:
        raise ValueError(f"No unique values found in column '{color_by}'.")

    px_colors = px.colors.qualitative.Plotly
    color_map = {val: px_colors[i % len(px_colors)] for i, val in enumerate(unique_values)}

    return color_map

###---------------------------------------------------
## PCA
def get_pca_plot(unselected_df, selected_df, x_col, y_col, group, point_size, color_map, visible_columns):
    print(f"    run get_pca_plot()")
    fig = go.Figure()

    # Unselected points (gray)
    fig.add_trace(go.Scattergl(
        x=unselected_df[x_col],
        y=unselected_df[y_col],
        mode='markers',
        marker=dict(color='lightgray', size=point_size, opacity=0.3),
        customdata=unselected_df[['id']],
        showlegend=False,
        hoverinfo='skip'  # Optional: speed up by disabling hover
    ))

    # Selected points, grouped by `group`
    if not selected_df.empty:
        # build hover columns: group first, then visible columns (excluding the group)
        hover_columns = [group] + [c for c in (visible_columns or []) if c != group]
        if not hover_columns:
            hover_columns = ['id']

        for group_name, group_df in selected_df.groupby(group):
            customdata = group_df[hover_columns].values

            fig.add_trace(go.Scattergl(
                x=group_df[x_col],
                y=group_df[y_col],
                mode='markers',
                marker=dict(size=point_size, color=color_map.get(group_name, 'blue')),
                name=str(group_name),
                customdata=customdata,
                hovertemplate=get_hover_text(hover_columns)
            ))

    
    fig.update_layout(
        template='plotly_white',
        margin=dict(l=0, r=0, t=20, b=0),
        showlegend=True,
        xaxis_title=x_col,
        yaxis_title=y_col
    )

    return fig


##----------------------------------------------------
## map
def get_map_plot(unselected_df, selected_df, group, point_size, color_map, visible_columns):
    print(f"    run get_map_plot()")
    # Base map with unselected (gray)
    fig = px.scatter_map(
        unselected_df,
        lat=ANNOTATION_LAT,
        lon=ANNOTATION_LONG,
        custom_data=['id'],
        color_discrete_sequence=["lightgray"],
        opacity=0.3,
        zoom=2
    )
    fig.update_traces(marker=dict(size=point_size), showlegend=False)

    # Add selected trace
    if len(selected_df) > 0:
        # build hover columns: group first, then visible columns (excluding the group)
        hover_columns = [group] + [c for c in (visible_columns or []) if c != group]
        # fallback to show at least id if no visible_columns provided
        if not hover_columns:
            hover_columns = ['id']

        selected_traces = px.scatter_map(
            selected_df,
            lat=ANNOTATION_LAT,
            lon=ANNOTATION_LONG,
            color=group,
            color_discrete_map=color_map,
            custom_data=hover_columns,
            zoom=2
        ).update_traces(marker=dict(size=point_size)).data

        # apply hovertemplate to each selected trace using get_hover_text
        hover_template = get_hover_text(hover_columns)
        for trace in selected_traces:
            trace.update(hovertemplate=hover_template, hoverinfo='text')
            fig.add_trace(trace)

    fig.update_layout(
        mapbox_style='open-street-map',
        margin={'l': 0, 'r': 0, 't': 0, 'b': 0},
        showlegend=False
    )

    return fig

##----------------------------------------------------
## time histogram
def get_time_histogram(all_df, selected_df):
    print(f"    run get_time_histogram()")
    # All points
    all_hist = px.histogram(
        all_df,
        x=ANNOTATION_TIME,
        nbins=args.time_hist_nbins,
        opacity=0.3,
        color_discrete_sequence=['lightgray'],
        labels={ANNOTATION_TIME: 'Time'}
    )

    # Selected points
    if not selected_df.empty:
        selected_hist = px.histogram(
            selected_df,
            x=ANNOTATION_TIME,
            nbins=args.time_hist_nbins,
            opacity=0.9,
            color_discrete_sequence=['#1F77B4'],
            labels={ANNOTATION_TIME: 'Time'}
        )

        fig = go.Figure(data=all_hist.data + selected_hist.data)

        start = int(selected_df[ANNOTATION_TIME].min())
        end = int(selected_df[ANNOTATION_TIME].max())

    else:
        fig = go.Figure(data=all_hist.data)

        start = int(all_df[ANNOTATION_TIME].min())
        end = int(all_df[ANNOTATION_TIME].max())


    fig.update_layout(
        template='plotly_white',
        showlegend=False,
        title="",
        barmode='overlay',  # <-- ensures overlay, not side-by-side
        xaxis=dict(
            title=f"Time range from {start:,} to {end:,}",
            rangeslider=dict(visible=True),
            type="linear"
        ),
        uirevision='time-histogram'
    )
    return fig

## time scatter
def get_time_scatter(unselected_df, selected_df, group, point_size, color_map, visible_columns):
    print(f"    run get_time_scatter()")

    # Jitter for y-axis
    def jitter(n, scale=1.0):
        return np.random.uniform(-scale, scale, n)

    fig = go.Figure()

    # Unselected points (gray)
    fig.add_trace(go.Scattergl(
        x=unselected_df[ANNOTATION_TIME],
        y=jitter(len(unselected_df), scale=0.5),
        mode='markers',
        marker=dict(color='lightgray', size=point_size, opacity=0.3),
        customdata=unselected_df[['id']],
        showlegend=False,
        hoverinfo='skip'  # Optional: speed up by disabling hover
    ))

    # Selected points, grouped by `group`
    if not selected_df.empty:
        # build hover columns: group first, then visible columns (excluding the group)
        hover_columns = [group] + [c for c in (visible_columns or []) if c != group]
        if not hover_columns:
            hover_columns = ['id']

        for group_name, group_df in selected_df.groupby(group):
            customdata = group_df[hover_columns].values

            fig.add_trace(go.Scattergl(
                x=group_df[ANNOTATION_TIME],
                y=jitter(len(group_df), scale=0.5),
                mode='markers',
                marker=dict(size=point_size, color=color_map.get(group_name, 'blue')),
                name=str(group_name),
                customdata=customdata,
                hovertemplate=get_hover_text(hover_columns)
            ))

        start = int(selected_df[ANNOTATION_TIME].min())
        end = int(selected_df[ANNOTATION_TIME].max())

    else:
        start = int(all_df[ANNOTATION_TIME].min())
        end = int(all_df[ANNOTATION_TIME].max())

    fig.update_layout(
        template='plotly_white',
        showlegend=False,
        title="",
        xaxis=dict(
            title=f"Time range from {start:,} to {end:,}",
            rangeslider=dict(visible=True),
            type="linear"
        ),
        yaxis=dict(
            title="Jitter",
            showticklabels=False,
            zeroline=False
        ),
        uirevision='time-histogram'
    )
    return fig


## -------------------------------------------------------------------
### callbacks

##----------------------------------------------------
## PCA

# PCA, Map, and Time Histogram callback (safe for missing components)
outputs = [Output("pca-plot", "figure")]
if ANNOTATION_LAT is not None and ANNOTATION_LONG is not None:
    outputs.append(Output("map-plot", "figure"))
if ANNOTATION_TIME is not None:
    outputs.append(Output("time-histogram", "figure"))
outputs.append(Output("selection-count", "children"))

# Register callback
@app.callback(
    outputs,
    [
        Input('x-axis-selector', 'value'),
        Input('y-axis-selector', 'value'),
        Input('group-selector', 'value'),
        Input('point-size-selector', 'value'),
        Input('selected-indices', 'data'),
        Input('visible-columns', 'data'),
    ],
    prevent_initial_call=True
)
def update_figures(x_col, y_col, group, point_size, selected_ids, visible_columns):
    triggered = [t['prop_id'] for t in callback_context.triggered]

    selected_df, unselected_df = split_selected_unselected(df, selected_ids)

    # Check if group exists
    color_map = get_color_map(df, group) if group in df.columns else {}

    results = []

    # PCA plot
    if 'pca-plot' not in triggered[0]:
        pca_fig = get_pca_plot(unselected_df, selected_df, x_col, y_col, group, point_size, color_map, visible_columns)
    else:
        pca_fig = dash.no_update
    results.append(pca_fig)


    # Map plot
    if (ANNOTATION_LAT is not None or ANNOTATION_LONG is not None):
        if any(
            k in triggered[0]
            for k in ['selected-indices', 'visible-columns', 'group-selector', 'point-size-selector']
            ) and 'map-plot' not in triggered[0] and (ANNOTATION_LAT is not None or ANNOTATION_LONG is not None):
            map_fig = get_map_plot(unselected_df, selected_df, group, point_size, color_map, visible_columns)
        else:
            map_fig = dash.no_update
        results.append(map_fig)

    # time histogram
    if ANNOTATION_TIME is not None:
        if 'time-histogram' not in triggered[0] and 'selected-indices' in triggered[0]:
            if args.time_hist:
                time_hist = get_time_histogram(df, selected_df)
            else:
                time_hist = get_time_scatter(unselected_df, selected_df, group, point_size, color_map, visible_columns)
        else:
            time_hist = dash.no_update
        results.append(time_hist)

    # Selection count
    if 'selected-indices' in triggered[0]:
        total = len(df)
        selected = len(selected_ids)
        selection_text = f"{selected} of {total} points selected / {len(visible_columns)} columns visible"
    else:
        selection_text = dash.no_update
    results.append(selection_text)

    return results


### ---------------------------------------------------
# Time histogram → selection callback
@app.callback(
    Output('selected-indices', 'data', allow_duplicate=True),
    Input('time-histogram', 'relayoutData'),
    prevent_initial_call=True
)
def filter_by_time_range(relayout_data):
    if ANNOTATION_TIME is None:
        return dash.no_update

    if not relayout_data:
        return df['id'].tolist()

    x_range = relayout_data.get('xaxis.range', None)
    if x_range is None:
        x0 = relayout_data.get('xaxis.range[0]')
        x1 = relayout_data.get('xaxis.range[1]')
        if x0 is not None and x1 is not None:
            x_range = [x0, x1]

    if x_range:
        filtered_df = df[(df[ANNOTATION_TIME] >= x_range[0]) & (df[ANNOTATION_TIME] <= x_range[1])]
        return filtered_df['id'].tolist()

    return df['id'].tolist()


### ---------------------------------------------------
# Column selection callback
@app.callback(
    Output('visible-columns', 'data', allow_duplicate=True),
    Input('annotation-table', 'selectedRows'),
    prevent_initial_call=True
)
def update_visible_columns(selected_rows):
    if not selected_rows:
        return ['id']
    selected_columns = ['id'] + [row['Abbreviation'] for row in selected_rows if row['Abbreviation'] != 'id']
    return selected_columns


### ---------------------------------------------------
# Data table columns update
@app.callback(
    Output('data-table', 'columnDefs', allow_duplicate=True),
    Input('visible-columns', 'data'),
    prevent_initial_call=True
)
def update_data_table_columns(selected_columns):
    if not selected_columns:
        return dash.no_update

    defs = [{'headerName': col, 'field': col, 'sortable': True, 'resizable': True, 
            **({'checkboxSelection': True, 'headerCheckboxSelection': True} if col == 'id' else {})} 
            for col in selected_columns]
    
    return defs


### ---------------------------------------------------
# Filter textarea → selection
@app.callback(
    Output('selected-indices', 'data', allow_duplicate=True),
    Input('texarea-expression', 'value'),
    prevent_initial_call=True
)
def filter_callback(query_str):
    if annotation is None:
        return dash.no_update

    # empty input -> select all
    if not query_str or str(query_str).strip() == "":
        return df['id'].tolist()
    try:
        # Use pandas eval for simple expressions, fallback to query for complex ones
        # This is faster for simple filters
        if "==" in query_str or ">" in query_str or "<" in query_str:
            filtered_df = df.eval(query_str)
            return df[filtered_df]['id'].tolist()
        else:
            filtered_df = df.query(query_str)
            return filtered_df['id'].tolist()
    except Exception as e:
        print(f"Query failed: {e}")
        return df['id'].tolist()




## Launch server
if __name__ == '__main__':
    print(f"Dashboard running at http://localhost:{args.server_port}")
    app.run(debug=True, port=args.server_port)


Dashboard running at http://localhost:8050


[2025-09-08 17:09:50,911] ERROR in app: Exception on /_dash-update-component [POST]
Traceback (most recent call last):
  File "/opt/anaconda3/envs/pyt_3_13_5/lib/python3.13/site-packages/pandas/core/indexes/base.py", line 3812, in get_loc
    return self._engine.get_loc(casted_key)
           ~~~~~~~~~~~~~~~~~~~~^^^^^^^^^^^^
  File "pandas/_libs/index.pyx", line 167, in pandas._libs.index.IndexEngine.get_loc
  File "pandas/_libs/index.pyx", line 196, in pandas._libs.index.IndexEngine.get_loc
  File "pandas/_libs/hashtable_class_helper.pxi", line 7088, in pandas._libs.hashtable.PyObjectHashTable.get_item
  File "pandas/_libs/hashtable_class_helper.pxi", line 7096, in pandas._libs.hashtable.PyObjectHashTable.get_item
KeyError: None

The above exception was the direct cause of the following exception:

Traceback (most recent call last):
  File "/opt/anaconda3/envs/pyt_3_13_5/lib/python3.13/site-packages/flask/app.py", line 917, in full_dispatch_request
    rv = self.dispatch_request()
  F

    run get_pca_plot()


[2025-09-08 17:17:38,160] ERROR in app: Exception on /_dash-update-component [POST]
Traceback (most recent call last):
  File "/opt/anaconda3/envs/pyt_3_13_5/lib/python3.13/site-packages/pandas/core/indexes/base.py", line 3812, in get_loc
    return self._engine.get_loc(casted_key)
           ~~~~~~~~~~~~~~~~~~~~^^^^^^^^^^^^
  File "pandas/_libs/index.pyx", line 167, in pandas._libs.index.IndexEngine.get_loc
  File "pandas/_libs/index.pyx", line 196, in pandas._libs.index.IndexEngine.get_loc
  File "pandas/_libs/hashtable_class_helper.pxi", line 7088, in pandas._libs.hashtable.PyObjectHashTable.get_item
  File "pandas/_libs/hashtable_class_helper.pxi", line 7096, in pandas._libs.hashtable.PyObjectHashTable.get_item
KeyError: None

The above exception was the direct cause of the following exception:

Traceback (most recent call last):
  File "/opt/anaconda3/envs/pyt_3_13_5/lib/python3.13/site-packages/flask/app.py", line 917, in full_dispatch_request
    rv = self.dispatch_request()
  F

    run get_pca_plot()


[2025-09-09 08:10:00,795] ERROR in app: Exception on /_dash-update-component [POST]
Traceback (most recent call last):
  File "/opt/anaconda3/envs/pyt_3_13_5/lib/python3.13/site-packages/pandas/core/indexes/base.py", line 3812, in get_loc
    return self._engine.get_loc(casted_key)
           ~~~~~~~~~~~~~~~~~~~~^^^^^^^^^^^^
  File "pandas/_libs/index.pyx", line 167, in pandas._libs.index.IndexEngine.get_loc
  File "pandas/_libs/index.pyx", line 196, in pandas._libs.index.IndexEngine.get_loc
  File "pandas/_libs/hashtable_class_helper.pxi", line 7088, in pandas._libs.hashtable.PyObjectHashTable.get_item
  File "pandas/_libs/hashtable_class_helper.pxi", line 7096, in pandas._libs.hashtable.PyObjectHashTable.get_item
KeyError: None

The above exception was the direct cause of the following exception:

Traceback (most recent call last):
  File "/opt/anaconda3/envs/pyt_3_13_5/lib/python3.13/site-packages/flask/app.py", line 917, in full_dispatch_request
    rv = self.dispatch_request()
  F

    run get_pca_plot()
